In [1]:
import itertools
import pandas as pd
import nusmv_utils
import fret_utils
import spot_utils
from tqdm import tqdm
import re
import json
import math
import collections
import numpy as np

In [2]:
mode_exp = "MODE_EXP"
condition_exp = "CONDITION_EXP"
stop_condition_exp = "STOP_CONDITION_EXP"
res_exp = "RES_EXP"

In [3]:
scope_mode_list = [mode_exp]
scope_options = fret_utils.generate_all_scope_options(scope_mode_list)

In [4]:
condition_exp_list = [condition_exp]
condition_options = fret_utils.generate_all_condition_options(condition_exp_list)

In [5]:
component_options = ["COMPONENT"]

In [6]:
obs_timing_list = []
timing_exp_list = [stop_condition_exp]
timing_options = fret_utils.generate_all_timing_options(timing_exp_list,additional_options=obs_timing_list)

In [7]:
len(scope_options)*len(condition_options)*len(timing_options)

660

In [8]:
response_options = [res_exp]

In [9]:
var_dict = {}
for var in [mode_exp,condition_exp,stop_condition_exp, res_exp]:
    var_dict[var] = "boolean"
var_dict

{'MODE_EXP': 'boolean',
 'CONDITION_EXP': 'boolean',
 'STOP_CONDITION_EXP': 'boolean',
 'RES_EXP': 'boolean'}

In [10]:
fretish_to_ltl = fret_utils.get_fretish_to_ltl_dict(scope_options,condition_options,component_options,timing_options,response_options)
len(fretish_to_ltl)

660

In [11]:
all_scope_option_list = []
for scope in scope_options:
    cur_fretish = fret_utils.get_fields_to_fretish(scope,condition='',component='COMPONENT',timing='immediately',response=res_exp)
    ltl_str,err_str = fret_utils.call_fret_formalize(cur_fretish)
    all_scope_option_list.append(ltl_str)
    #print(get_align_mask_options(ltl_str,res_exp))

In [12]:
all_cond_option_list = []
for condition in condition_options:
    cur_fretish = fret_utils.get_fields_to_fretish(scope="",condition=condition,component='COMPONENT',timing='immediately',response=res_exp)
    ltl_str,err_str = fret_utils.call_fret_formalize(cur_fretish)
    all_cond_option_list.append(ltl_str)
    #print(get_align_mask_options(ltl_str,res_exp))

In [13]:
all_timing_option_list = []
for timing in timing_options:
    cur_fretish = fret_utils.get_fields_to_fretish(scope="",condition="",component='COMPONENT',timing=timing,response=res_exp)
    ltl_str,err_str = fret_utils.call_fret_formalize(cur_fretish)
    all_timing_option_list.append(ltl_str)
    #print(timing,get_align_mask_options(ltl_str,res_exp))

In [14]:
"""
def get_proxy(timing,condition=None,res_exp="RES_EXP",stop_condition_exp=None,condition_exp=None):
    if condition is None:
        #timing proxy under any scope+condition
        if "until" not in timing and "before" not in timing:
            timing_fretish = fret_utils.get_fields_to_fretish(scope="",condition="",component='COMPONENT',timing=timing,response=res_exp)
            #timing_ltl = fretish_to_ltl[timing_fretish]
            timing_ltl,_ = fret_utils.call_fret_formalize(timing_fretish)
            proxy = [timing_ltl]
        elif "until" in timing:
            weak_until = f"(({res_exp}) U ({stop_condition_exp}))" #strong until
            strong_until = f"(({stop_condition_exp}) V (({stop_condition_exp}) | ({res_exp})))" #weak until
            proxy = [weak_until,strong_until]
        elif "before" in timing:
            #strong_before = f"((({res_exp}) V (!({stop_condition_exp}))) & F ({stop_condition_exp}))" #strong before
            strong_before = f"((({res_exp}) V (!({stop_condition_exp}))) & F ({res_exp}))" #strong before
            weak_before = f"(({res_exp}) V (! ({stop_condition_exp})))" #weak before
            proxy = [strong_before,weak_before]
    else:
        #condition+timing proxy under any scope
        timing_proxy_list = get_proxy(timing=timing,condition=None,res_exp=res_exp,stop_condition_exp=stop_condition_exp)
        if "whenever" in condition:
            proxy = []
            for timing_proxy in timing_proxy_list:
                proxy.append(f"!(G (({condition_exp}) -> !({timing_proxy})))")
                proxy.append(f"(G (({condition_exp}) -> ({timing_proxy})))")
            proxy.append(f"!({condition_exp})")
        elif "upon" in condition:
            proxy = []
            for timing_proxy in timing_proxy_list:
                proxy.append(f"!((G (((! ({condition_exp})) & (X ({condition_exp}))) -> (X !({timing_proxy}))) & (({condition_exp}) -> !({timing_proxy}))))")
                proxy.append(f"((G (((! ({condition_exp})) & (X ({condition_exp}))) -> (X ({timing_proxy}))) & (({condition_exp}) -> ({timing_proxy}))))")
            proxy.append(f"!({condition_exp})")
        elif condition == "":
            proxy = timing_proxy_list
    return proxy
"""

'\ndef get_proxy(timing,condition=None,res_exp="RES_EXP",stop_condition_exp=None,condition_exp=None):\n    if condition is None:\n        #timing proxy under any scope+condition\n        if "until" not in timing and "before" not in timing:\n            timing_fretish = fret_utils.get_fields_to_fretish(scope="",condition="",component=\'COMPONENT\',timing=timing,response=res_exp)\n            #timing_ltl = fretish_to_ltl[timing_fretish]\n            timing_ltl,_ = fret_utils.call_fret_formalize(timing_fretish)\n            proxy = [timing_ltl]\n        elif "until" in timing:\n            weak_until = f"(({res_exp}) U ({stop_condition_exp}))" #strong until\n            strong_until = f"(({stop_condition_exp}) V (({stop_condition_exp}) | ({res_exp})))" #weak until\n            proxy = [weak_until,strong_until]\n        elif "before" in timing:\n            #strong_before = f"((({res_exp}) V (!({stop_condition_exp}))) & F ({stop_condition_exp}))" #strong before\n            strong_before =

In [15]:
#verify timing proxies under any scope+condition
total_var_dict = var_dict.copy()
all_valid_proxy_array = []
all_proxy_list = [fret_utils.get_proxy(timing,condition=None,res_exp=res_exp,stop_condition_exp=stop_condition_exp) for timing in timing_options]
for scope in scope_options:
    if scope == f"while {mode_exp}":
        scope_extra_constraint = f"(G ({mode_exp}))"
    elif scope == f"when not in {mode_exp}":
        scope_extra_constraint = f"(G ! ({mode_exp}))"
    elif scope == f"only while {mode_exp}":
        scope_extra_constraint = f"(G ! ({mode_exp}))"
    elif scope == f"only after {mode_exp}":
        scope_extra_constraint = f"(G ({mode_exp}))"
    elif scope == f"after {mode_exp}":
        scope_extra_constraint = f"(({mode_exp}) & X G ! ({mode_exp}))"
    elif scope == f"only before {mode_exp}":
        scope_extra_constraint = f"(G ({mode_exp}))"
    elif scope == f"before {mode_exp}":
        scope_extra_constraint = f"(G !({mode_exp}))"
    elif "whenever" in scope or "upon" in scope:
        scope_extra_constraint = f"(({mode_exp})) & X (G ! ({mode_exp}))"    
    else:
        scope_extra_constraint = "TRUE"
    for condition in condition_options:
        if scope == f"after {mode_exp}" and ("whenever" in condition or "upon" in condition):
            extra_constraint = f"{scope_extra_constraint} & X (({condition_exp})) & X X (G ! ({condition_exp}))"
        elif "whenever" in condition or "upon" in condition:
            extra_constraint = f"{scope_extra_constraint} & (({condition_exp})) & X (G ! ({condition_exp}))"
        else:
            extra_constraint = scope_extra_constraint
        valid_proxy_array = []
        cur_f_list = []
        for t_idx in range(len(timing_options)):
            timing = timing_options[t_idx]
            cur_fretish = fret_utils.get_fields_to_fretish(scope=scope,condition=condition,component='COMPONENT',timing=timing,response=res_exp)
            cur_ltl = fretish_to_ltl[cur_fretish]
            cur_f_list.append(cur_ltl)
            #proxy_list = get_proxy(timing,condition)
            proxy_list = all_proxy_list[t_idx]
            proxy_hold = " & ".join(proxy_list)
            proxy_nothold = " & ".join([f"!({e})" for e in proxy_list])
            is_proxyhold_list = []
            is_proxynothold_list = []
            for i in range(2):
                cur_timestep = "X "*i
                proxyhold_implies_cur = f"({extra_constraint}) -> (({cur_timestep}({proxy_hold}) -> ({cur_ltl})) & ({cur_timestep}({proxy_nothold}) -> !({cur_ltl})))"
                proxynothold_implies_cur = f"({extra_constraint}) -> (({cur_timestep}({proxy_nothold}) -> ({cur_ltl})) & ({cur_timestep}({proxy_hold}) -> !({cur_ltl})))"   
                is_proxyhold_implies_cur = nusmv_utils.get_nusmv_ltl_true(total_var_dict,proxyhold_implies_cur) == None
                is_proxynothold_implies_cur = nusmv_utils.get_nusmv_ltl_true(total_var_dict,proxynothold_implies_cur) == None
                is_proxyhold_list.append(is_proxyhold_implies_cur)
                is_proxynothold_list.append(is_proxynothold_implies_cur)
            assert any(is_proxyhold_list) or any(is_proxynothold_list), " ".join([scope,condition,timing])
            valid_proxy_array.append([is_proxyhold_list,is_proxynothold_list])
        assert np.max(np.prod(valid_proxy_array,axis=0).astype(bool)) == True,scope + condition
        all_valid_proxy_array.append(valid_proxy_array)
all_valid_proxy_array = np.array(all_valid_proxy_array,dtype=bool) #template X proxy X hold/not hold X timestep
for template_idx in range(len(all_valid_proxy_array)):
    #take the product to determine a hold/not hold and timestep that works for all proxies
    #take max to determine if the condition can be satisfied
    assert np.max(np.prod(all_valid_proxy_array[template_idx],axis=0).astype(bool)) == True

In [16]:
#verify condition+timing proxies under any scope
all_proxy_list = []
for condition in condition_options:
    all_proxy_list.append([])
    for timing in timing_options:
        all_proxy_list[-1].append(fret_utils.get_proxy(timing,condition=condition,
                                            res_exp=res_exp,stop_condition_exp=stop_condition_exp,condition_exp=condition_exp))
        
all_valid_proxy_array = []
for scope in scope_options:
    if scope == f"while {mode_exp}":
        scope_extra_constraint = f"(G ({mode_exp}))"
    elif scope == f"when not in {mode_exp}":
        scope_extra_constraint = f"(G ! ({mode_exp}))"
    elif scope == f"only while {mode_exp}":
        scope_extra_constraint = f"(G ! ({mode_exp}))"
    elif scope == f"only after {mode_exp}":
        scope_extra_constraint = f"(G ({mode_exp}))"
    elif scope == f"after {mode_exp}":
        scope_extra_constraint = f"(({mode_exp}) & X G ! ({mode_exp}))"
    elif scope == f"only before {mode_exp}":
        scope_extra_constraint = f"(G ({mode_exp}))"
    elif scope == f"before {mode_exp}":
        scope_extra_constraint = f"(G !({mode_exp}))"
    elif "whenever" in scope:
        #scope_extra_constraint = f"(G ({mode_exp}))" 
        scope_extra_constraint = f"((({mode_exp})) & X (G ! ({mode_exp})))"  
    elif "upon" in scope:
        scope_extra_constraint = f"(G ({mode_exp}))"  
    else:
        scope_extra_constraint = "TRUE"
    valid_proxy_array = []
    for c_idx in range(len(condition_options)):
        condition = condition_options[c_idx]
        for t_idx in range(len(timing_options)):
            timing = timing_options[t_idx]
            cur_fretish = fret_utils.get_fields_to_fretish(scope=scope,condition=condition,component='COMPONENT',timing=timing,response=res_exp)
            cur_ltl = fretish_to_ltl[cur_fretish]
            #proxy_list = get_proxy(timing,condition)
            proxy_list = all_proxy_list[c_idx][t_idx]
            proxy_hold = " & ".join(proxy_list)
            proxy_nothold = " & ".join([f"!({e})" for e in proxy_list])
            assert nusmv_utils.get_nusmv_ltl_satisfiable(total_var_dict,proxy_hold) is not None
            assert nusmv_utils.get_nusmv_ltl_satisfiable(total_var_dict,proxy_nothold) is not None
            is_proxyhold_list = []
            is_proxynothold_list = []
            for i in range(2):
                cur_timestep = "X "*i
                proxyhold_implies_cur =    f"({scope_extra_constraint}) -> (({cur_timestep}({proxy_hold})    -> !({cur_ltl})) )"# & ({cur_timestep}({proxy_nothold}) -> !({cur_ltl})))"
                proxynothold_implies_cur = f"({scope_extra_constraint}) -> (({cur_timestep}({proxy_nothold}) -> !({cur_ltl})) )"# & ({cur_timestep}({proxy_hold}) -> !({cur_ltl})))"   
                is_proxyhold_implies_cur = nusmv_utils.get_nusmv_ltl_true(total_var_dict,proxyhold_implies_cur) == None
                is_proxynothold_implies_cur = nusmv_utils.get_nusmv_ltl_true(total_var_dict,proxynothold_implies_cur) == None
                is_proxyhold_list.append(is_proxyhold_implies_cur)
                is_proxynothold_list.append(is_proxynothold_implies_cur)
            if not (any(is_proxyhold_list) or any(is_proxynothold_list)):
                print(" ".join([scope,condition,timing]))
                #print(proxyhold_implies_cur)
            valid_proxy_array.append([is_proxyhold_list,is_proxynothold_list])
    valid_proxy_array = np.array(valid_proxy_array) #proxy X hold/not hold X timestep
    assert np.max(np.prod(valid_proxy_array,axis=0).astype(bool)) == True," ".join([scope,condition,timing])
    all_valid_proxy_array.append(valid_proxy_array)
all_valid_proxy_array = np.array(all_valid_proxy_array,dtype=bool) #template X proxy X hold/not hold X timestep
for template_idx in range(len(all_valid_proxy_array)):
    #take the product to determine a hold/not hold and timestep that works for all proxies
    #take max to determine if the condition can be satisfied
    assert np.max(np.prod(all_valid_proxy_array[template_idx],axis=0).astype(bool)) == True